# Implementing backward elimination

## Step 1: Import the required libraries
Before starting, make sure you have the necessary libraries installed. You will be using Python along with the following libraries:
* pandas will be used to handle the dataset.
* statsmodels will help you perform statistical modeling, which is required for backward elimination.

In [1]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

## Step 2: Load and prepare the data
You’ll use a sample dataset to predict whether a student passes a speculative future assignment (not shown) based on their study hours and previous exam scores. Alternatively, you can apply the same steps to your own dataset:
In this example, StudyHours and PrevExamScore are the features, and Pass is the target variable (0 = Fail, 1 = Pass).

In [2]:
# Sample dataset
data = {
    'StudyHours': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'PrevExamScore': [30, 40, 45, 50, 60, 65, 70, 75, 80, 85],
    'Pass': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]  # 0 = Fail, 1 = Pass
}

df = pd.DataFrame(data)

# Features and target variable
X = df[['StudyHours', 'PrevExamScore']]
y = df['Pass']

## Step 3: Add a constant to the model
In statsmodels, you need to add a constant to your feature matrix for the intercept term. This constant will be necessary for the linear regression model used in backward elimination.

In [3]:
# Add a constant to the model (for the intercept)
X = sm.add_constant(X)

## Step 4: Fit the initial model
Now, you will fit the initial model using all the available features.

The goal is to start with all features and then progressively remove the least significant ones. The output will show a summary of the model, including the p-values for each feature. The p-value helps you determine the statistical significance of each feature: features with high p-values are considered less significant and should be removed.

In [4]:
# Fit the model using Ordinary Least Squares (OLS) regression
model = sm.OLS(y, X).fit()

# Display the summary, including p-values for each feature
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   Pass   R-squared:                       0.758
Model:                            OLS   Adj. R-squared:                  0.688
Method:                 Least Squares   F-statistic:                     10.94
Date:                Wed, 07 Jan 2026   Prob (F-statistic):            0.00701
Time:                        13:48:14   Log-Likelihood:               -0.17258
No. Observations:                  10   AIC:                             6.345
Df Residuals:                       7   BIC:                             7.253
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.3333      1.464     -0.228

## Step 5: Implement backward elimination
The main idea behind backward elimination is to iteratively remove the feature with the highest p-value—greater than 0.05 in this case—and refit the model until all remaining features have a p-value less than 0.05.

Step-by-step process:

1. Fit the model with all features.
2. Identify the feature with the highest p-value.
3. Remove the feature with the highest p-value.
4. Refit the model and repeat until all remaining features are statistically significant.

Here’s a simple implementation of this process:

In [5]:
# Define a significance level
significance_level = 0.05

# Perform backward elimination
while True:
    # Fit the model
    model = sm.OLS(y, X).fit()
    # Get the highest p-value in the model
    max_p_value = model.pvalues.max()
    
    # Check if the highest p-value is greater than the significance level
    if max_p_value > significance_level:
        # Identify the feature with the highest p-value
        feature_to_remove = model.pvalues.idxmax()
        print(f"Removing feature: {feature_to_remove} with p-value: {max_p_value}")
        
        # Drop the feature
        X = X.drop(columns=[feature_to_remove])
    else:
        break

# Display the final model summary
print(model.summary())

Removing feature: PrevExamScore with p-value: 0.9999999999999999
Removing feature: const with p-value: 0.11419580126842226
                                 OLS Regression Results                                
Dep. Variable:                   Pass   R-squared (uncentered):                   0.831
Model:                            OLS   Adj. R-squared (uncentered):              0.812
Method:                 Least Squares   F-statistic:                              44.31
Date:                Wed, 07 Jan 2026   Prob (F-statistic):                    9.31e-05
Time:                        13:51:18   Log-Likelihood:                         -1.8294
No. Observations:                  10   AIC:                                      5.659
Df Residuals:                       9   BIC:                                      5.961
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                